# Validate Before You Estimate: Links as Measurements

* * *

<div class="alert alert-success">

### Learning Objectives

After this notebook, you will be able to:

- Explain why record linkage is a classification problem with the pair as the unit.
- Use rules for the easy pairs and a model (or a person) for the ambiguous band.
- Measure precision and recall of a linkage policy against known truth.
- Show how linkage error carries into a downstream estimate.
</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
🥊 **Challenge**: Interactive exercise. We'll work through these in the workshop!<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning**: Heads-up about tricky stuff or common mistakes.

### Sections
1. [The Data](#data)
2. [Rules First](#rules)
3. [The Model Adjudicates the Band](#model)
4. [Downstream](#downstream)
5. [Key Points](#keypoints)

The rule-based step is in `scripts/04_linkage_rules.py`; the model's decisions on the ambiguous pairs were run beforehand with `scripts/05_adjudicate_pairs.py` and saved in `outputs/`.

In [1]:
import numpy as np
import pandas as pd

<a id='data'></a>

# The Data

Febrl ships two files of 5,000 person records. File B is a corrupted copy of file A (typos, swapped digits, moved addresses), so every true link is known. We keep four fields two administrative files would realistically share: given name, surname, date of birth, postcode.

In [2]:
scores = pd.read_csv("../outputs/linkage_scores.csv")
print(scores.shape)
scores.head()

(137663, 9)


,rec_id_1,rec_id_2,given_name,surname,date_of_birth,postcode,score,truth,zone
0,rec-0-org,rec-0-dup-0,1.0,1.0,1,1,4.0,1,match
1,rec-0-org,rec-1505-dup-0,0.0,1.0,0,0,1.0,0,non-match
2,rec-0-org,rec-1636-dup-0,0.0,1.0,0,0,1.0,0,non-match
3,rec-0-org,rec-2074-dup-0,0.0,1.0,0,0,1.0,0,non-match
4,rec-0-org,rec-2526-dup-0,0.0,0.0,0,1,1.0,0,non-match


<a id='rules'></a>

# Rules First

Each candidate pair got one point per agreeing field. Four points is a sure match, zero or one a sure non-match, and the band at two is **ambiguous**.

In [3]:
pd.crosstab(scores["score"], scores["truth"])

truth,0,1
score,,
0.0,14514,0
1.0,117271,96
2.0,890,593
3.0,1,1553
4.0,0,2745


## 🥊 Notebook Challenge 1: Precision and Recall of the Rules

Compute precision and recall when the ambiguous band is (a) dropped and (b) kept as matches. There are 5,000 true links in total.

In [4]:
N_TRUE = 5000
sure = scores[scores["zone"] == "match"]
band = scores[scores["zone"] == "ambiguous"]

def precision_recall(pred):
    tp = int(pred["truth"].sum())
    return tp / len(pred), tp / N_TRUE

for name, pred in [("drop the band", sure), ("keep the band", pd.concat([sure, band]))]:
    p, r = precision_recall(pred)
    print(f"{name:14s} precision {p:.3f}  recall {r:.3f}")

drop the band  precision 1.000  recall 0.860
keep the band  precision 0.846  recall 0.978


<a id='model'></a>

# The Model Adjudicates the Band

Each ambiguous pair was shown to the model as two short records, with the instruction to decide whether they refer to the same person.

In [5]:
print(open("../prompts/pairs_prompt.txt").read())

Two administrative records may refer to the same person. The records come from different files, were typed by different clerks, and may contain typos, missing values, swapped fields, or a changed address. They share no identifier.

Record A: {record_a}
Record B: {record_b}

Decide whether the two records refer to the same person. Treat small spelling differences, transposed digits, nicknames, and a different postcode with everything else matching as compatible with the same person. Treat a different date of birth together with a different name as different people.

Answer with a JSON object of the form {"match": "yes" or "no", "why": "<one short sentence>"} and nothing else.



The cached decisions are one row per pair and model. We attach the truth and the two postcodes, and turn the model's `yes` into a 1.

In [6]:
pairs = pd.read_csv("../outputs/llm_pairs_clean.csv").drop_duplicates(subset=["model", "pair_id"], keep="last")
amb = pd.read_csv("../outputs/linkage_ambiguous_pairs.csv", dtype={"pair_id": str})
pairs = pairs.merge(amb[["pair_id", "truth", "postcode_a", "postcode_b"]], on="pair_id")
pairs["pred"] = (pairs["match"] == "yes").astype(int)
pairs.groupby("model")[["pred", "truth"]].mean().round(3)

,pred,truth
model,,
claude-haiku-5-5,0.299,0.4
claude-sonnet-5-5,0.272,0.4


Precision and recall inside the band, and then for the whole policy (rules plus model), with the 5,000 true links as the denominator of recall.

In [7]:
for model, g in pairs.groupby("model"):
    tp = int(((g["pred"] == 1) & (g["truth"] == 1)).sum())
    fp = int(((g["pred"] == 1) & (g["truth"] == 0)).sum())
    print(f"{model:20s} inside the band: precision {tp / (tp + fp):.3f}  recall {tp / g['truth'].sum():.3f}")
    tp_all = int(sure["truth"].sum()) + tp
    fp_all = int((sure["truth"] == 0).sum()) + fp
    print(f"{'':20s} rules + model:   precision {tp_all / (tp_all + fp_all):.3f}  recall {tp_all / N_TRUE:.3f}")

claude-haiku-5-5     inside the band: precision 0.944  recall 0.707
                     rules + model:   precision 0.995  recall 0.943
claude-sonnet-5-5    inside the band: precision 0.950  recall 0.646
                     rules + model:   precision 0.996  recall 0.936


🔔 **Question**: The stronger model is slightly more precise and finds fewer true pairs. Which error would you rather make, and does it depend on what you will estimate?

<a id='downstream'></a>

# Downstream

The research question: what share of people changed postcode between the two files? Postcode is also a matching field, so linkage errors are correlated with the outcome.

A pair "moved" when the postcode field disagrees. We compute the share who moved under each policy: among all 5,000 true links (including the 13 that blocking never paired, all of them movers), among the sure matches only, among sure matches plus the whole band, and among sure matches plus the pairs the model accepted.

In [8]:
from recordlinkage.datasets import load_febrl4

dfA, dfB, true_links = load_febrl4(return_links=True)
postcode_a = dfA.loc[true_links.get_level_values(0), "postcode"].astype(str).values
postcode_b = dfB.loc[true_links.get_level_values(1), "postcode"].astype(str).values
truth_moved = (postcode_a != postcode_b).mean()

scores["moved"] = (scores["postcode"] == 0).astype(int)
strict = scores.loc[sure.index, "moved"].mean()
lenient = scores.loc[pd.concat([sure, band]).index, "moved"].mean()
g = pairs[(pairs["model"] == "claude-haiku-5-5") & (pairs["pred"] == 1)]
model_moved = pd.concat([scores.loc[sure.index, "moved"],
                         (g["postcode_a"].astype(str) != g["postcode_b"].astype(str)).astype(int)]).mean()
pd.Series({"truth": truth_moved, "rules, strict": strict, "rules, lenient": lenient,
           "rules + model": model_moved}).round(3)

truth             0.156
rules, strict     0.122
rules, lenient    0.223
rules + model     0.142
dtype: float64

## 🥊 Notebook Challenge 2: Review a Sample of the Band

In a real project the truth is unknown. Draw 100 pairs from the band at random, pretend a research assistant reviewed them, and estimate the model's precision and recall inside the band from that sample alone. How close are you to the full-band numbers above?

⚠️ **Warning**: a sample of the band tells you about the band. The true links the rules already missed (score 1, below the band) need their own reviewed sample, or your recall will look better than it is.

In [9]:
band_model = pairs[pairs["model"] == "claude-haiku-5-5"]
reviewed = band_model.sample(100, random_state=7)
tp = int(((reviewed["pred"] == 1) & (reviewed["truth"] == 1)).sum())
fp = int(((reviewed["pred"] == 1) & (reviewed["truth"] == 0)).sum())
fn = int(((reviewed["pred"] == 0) & (reviewed["truth"] == 1)).sum())
print(f"from 100 reviewed pairs: precision {tp / (tp + fp):.2f}  recall {tp / (tp + fn):.2f}")

from 100 reviewed pairs: precision 0.96  recall 0.77


<a id='keypoints'></a>

<div class="alert alert-success">

### ❗ Key Points

- A link is a label on a pair, with error like any other label.
- Rules handle the easy pairs cheaply; the ambiguous band is where a person or a model has to decide.
- Measure precision and recall on a reviewed random sample of the band, by the groups you compare.
- Missed links and invented links push a downstream estimate in opposite directions; the right policy depends on the number you want to know.
</div>